# Week-6 Assignment (Part-2)

### Q1: Explain the roles of the Driver, Cluster Manager, and Executor in a Spark application.

### Answer

A Spark application consists of three main components:

- **Driver:** The Driver is the main program that creates the SparkSession, builds the execution plan, and coordinates all tasks.
- **Cluster Manager:** It manages the available resources in the cluster and assigns them to Spark applications.
- **Executors:** Executors run on worker nodes and perform the actual computations on the data partitions. They also store intermediate results and return the output to the Driver.

Together, these components allow Spark to process large datasets efficiently in a distributed environment.

### Q2: How does Spark's Lazy Evaluation strategy improve performance when chain-processing large datasets?

### Answer

Spark does not execute transformations immediately. Instead, it records all the transformations and waits until an action such as `show()` or `count()` is called.

This allows Spark to optimize the complete execution plan using a DAG before processing the data. As a result, unnecessary computations are reduced and the overall execution becomes more efficient.

### Q3: Write a Spark command to read a CSV file located at "data/source.csv", ensuring the first row is treated as a header and inferSchema is enabled. 

In [0]:
df = spark.read.option("header", True).option("inferSchema",True).csv("/Volumes/workspace/default/my_files/Student.csv")
df.show(10)

+---------+-------------+----------+----+------+---+----------+-----+----+---------+-----------+--------------------+
|StudentID|         Name|Department|Year|Gender|Age|Attendance|Marks|CGPA|     City|Scholarship|               Email|
+---------+-------------+----------+----+------+---+----------+-----+----+---------+-----------+--------------------+
|     1001| Aarav Sharma|        CS|   1|     M| 18|        92|   85| 8.8|    Delhi|        Yes|aarav.sharma@univ...|
|     1002|    Aditi Rao|       ECE|   2|     F| 19|        88|   78| 8.2|   Mumbai|         No|  aditi.rao@univ.edu|
|     1003|  Arjun Verma|        ME|   3|     M| 21|        74|   65| 7.1|Bangalore|         No|arjun.verma@univ.edu|
|     1004|  Ananya Iyer|        CS|   1|     F| 18|        95|   92| 9.4|  Chennai|        Yes|ananya.iyer@univ.edu|
|     1005| Aditya Patel|        IT|   4|     M| 22|        68|   58| 6.5|Ahmedabad|         No|aditya.patel@univ...|
|     1006|  Avani Singh|       ECE|   2|     F| 20|    

### Q4: What is the difference between CSV and Parquet in terms of storage (row-based vs. columnar) and why does it matter for performance? 

### Answer
CSV stores data row by row, whereas Parquet stores data column by column.

Because Parquet is columnar, Spark reads only the required columns instead of the entire file. It also supports compression and stores schema information, making it faster and more storage-efficient than CSV for analytics and big data processing.

### Q5:Given a DataFrame df, write a query to select the columns StudentID and Marks where the Department is 'CS'.

In [0]:
df.filter(df.Department == "CS").select("StudentID", "Marks").show(10)

+---------+-----+
|StudentID|Marks|
+---------+-----+
|     1001|   85|
|     1004|   92|
|     1007|   88|
|     1011|   95|
|     1015|   87|
|     1018|   91|
|     1022|   70|
|     1026|   73|
|     1030|   80|
|     1035|   88|
+---------+-----+
only showing top 10 rows


### Q6:Write the code to rename the column Name to Student_Name and cast the Marks column from String to Integer.

In [0]:
# using try_cast instead of cast coz marks contains null values if you use cast it throws an error 
from pyspark.sql.functions import col

updated_df = (df.withColumnRenamed("Name", "Student_Name").withColumn("Marks", col("Marks").try_cast("int")))

updated_df.printSchema()

root
 |-- StudentID: integer (nullable = true)
 |-- Student_Name: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Attendance: string (nullable = true)
 |-- Marks: integer (nullable = true)
 |-- CGPA: double (nullable = true)
 |-- City: string (nullable = true)
 |-- Scholarship: string (nullable = true)
 |-- Email: string (nullable = true)



### Q7: How does Spark use the Lineage Graph (DAG) to provide fault tolerance if a worker node fails? 

### Answer

Spark keeps track of every transformation using a Lineage Graph (DAG).

If an executor or worker node fails, Spark does not need to reload the entire dataset. Instead, it uses the DAG to identify the lost partition and recomputes only the required data from the original source. This provides fault tolerance without storing multiple copies of the data.

### Q8:Write a query to display students whose attendance is greater than or equal to 90 and marks are greater than 80.

In [0]:
df.filter((col("Attendance").try_cast("int") >= 90) &(col("Marks").try_cast("int") > 80)).show(10)

+---------+-------------+----------+----+------+---+----------+-----+----+---------+-----------+--------------------+
|StudentID|         Name|Department|Year|Gender|Age|Attendance|Marks|CGPA|     City|Scholarship|               Email|
+---------+-------------+----------+----+------+---+----------+-----+----+---------+-----------+--------------------+
|     1001| Aarav Sharma|        CS|   1|     M| 18|        92|   85| 8.8|    Delhi|        Yes|aarav.sharma@univ...|
|     1004|  Ananya Iyer|        CS|   1|     F| 18|        95|   92| 9.4|  Chennai|        Yes|ananya.iyer@univ.edu|
|     1008|  Amrita Nair|        EE|   1|     F| 18|        90|   81| 8.3|    Kochi|         No|amrita.nair@univ.edu|
|     1011| Bhavya Joshi|        CS|   4|     F| 22|        94|   95| 9.6|     Pune|        Yes|bhavya.joshi@univ...|
|     1015|   Devika Sen|        CS|   2|     F| 19|        91|   87| 8.7|  Kolkata|        Yes| devika.sen@univ.edu|
|     1018|   Divya Teja|        CS|   3|     F| 20|    

### Q9: Explain the concept of Predicate Pushdown in Parquet and how it affects the amount of data loaded into memory. 

Predicate Pushdown is an optimization available in formats such as Parquet.

When a filter condition is applied, Spark pushes the filter closer to the storage layer. Only the required rows are read into memory instead of scanning the complete dataset.

This reduces disk I/O and improves query performance.

###Q10: Write a code snippet to create a new column BonusMarks by adding 5 marks to the existing Marks column.

In [0]:
updated_df.withColumn("BonusMarks",col("Marks").cast("int") + 5).show(5)

+---------+------------+----------+----+------+---+----------+-----+----+---------+-----------+--------------------+----------+
|StudentID|Student_Name|Department|Year|Gender|Age|Attendance|Marks|CGPA|     City|Scholarship|               Email|BonusMarks|
+---------+------------+----------+----+------+---+----------+-----+----+---------+-----------+--------------------+----------+
|     1001|Aarav Sharma|        CS|   1|     M| 18|        92|   85| 8.8|    Delhi|        Yes|aarav.sharma@univ...|        90|
|     1002|   Aditi Rao|       ECE|   2|     F| 19|        88|   78| 8.2|   Mumbai|         No|  aditi.rao@univ.edu|        83|
|     1003| Arjun Verma|        ME|   3|     M| 21|        74|   65| 7.1|Bangalore|         No|arjun.verma@univ.edu|        70|
|     1004| Ananya Iyer|        CS|   1|     F| 18|        95|   92| 9.4|  Chennai|        Yes|ananya.iyer@univ.edu|        97|
|     1005|Aditya Patel|        IT|   4|     M| 22|        68|   58| 6.5|Ahmedabad|         No|aditya.pa

### Q11: What is the difference between Transformations and Actions? Provide two examples of each. 

**Transformations** create a new DataFrame but are not executed immediately.

Examples:
- filter()
- select()

**Actions** trigger the execution of all pending transformations and produce the final result.

Examples:
- show()
- count()

### Q12:Write the Spark command to load the Parquet file created in the previous assignment, remove rows where Email is null, and save the result as a CSV file.

In [0]:
df.filter(col("Email").isNotNull()).write.mode("overwrite").option("header", True).csv("/Volumes/workspace/default/my_files/output1.csv")

### Q13: In Spark Architecture, what is the difference between Client Mode and Cluster Mode? 

In **Client Mode**, the Driver runs on the user's machine, while the Executors run on the cluster.

In **Cluster Mode**, both the Driver and Executors run inside the cluster.

Cluster Mode is generally preferred for production because the application continues to run even if the client disconnects.

### Q14:Write a query to display students who belong to the CSE department OR have a CGPA greater than 8.5.

In [0]:
df.filter((col("Department") == "CSE") |(col("CGPA") > 8.5)).show(10)


+---------+-------------+----------+----+------+---+----------+-----+----+---------+-----------+--------------------+
|StudentID|         Name|Department|Year|Gender|Age|Attendance|Marks|CGPA|     City|Scholarship|               Email|
+---------+-------------+----------+----+------+---+----------+-----+----+---------+-----------+--------------------+
|     1001| Aarav Sharma|        CS|   1|     M| 18|        92|   85| 8.8|    Delhi|        Yes|aarav.sharma@univ...|
|     1004|  Ananya Iyer|        CS|   1|     F| 18|        95|   92| 9.4|  Chennai|        Yes|ananya.iyer@univ.edu|
|     1007|  Ayush Gupta|        CS|   3|     M| 21|        89|   88| 8.9|    Delhi|        Yes|ayush.gupta@univ.edu|
|     1011| Bhavya Joshi|        CS|   4|     F| 22|        94|   95| 9.6|     Pune|        Yes|bhavya.joshi@univ...|
|     1015|   Devika Sen|        CS|   2|     F| 19|        91|   87| 8.7|  Kolkata|        Yes| devika.sen@univ.edu|
|     1018|   Divya Teja|        CS|   3|     F| 20|    

### Q15: When exploring a dataset, why is it safer to use .show(5) instead of .collect() on a multi-terabyte dataset? 

The `show(5)` function displays only a small number of rows and is mainly used for inspecting data.

The `collect()` function brings all the data from every partition to the Driver. For very large datasets, this can consume a large amount of memory and may cause the application to fail.

Therefore, `show(5)` is much safer when exploring large datasets.